In [1]:
# Ячейка 1
import os
import sys
import random
import subprocess
from typing import List, Dict, Optional, Tuple

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Фиксируем SEED
def set_seed(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)
    try:
        import torch
        torch.manual_seed(seed)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(seed)
    except ImportError:
        pass

set_seed(42)

# Функция для автоматической установки пакетов (на всякий случай)
def ensure_package(package_name: str, import_name: Optional[str] = None) -> None:
    target = import_name or package_name
    try:
        __import__(target)
    except ImportError:
        print(f"Устанавливаем пакет: {package_name}")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package_name])

# Устанавливаем и импортируем все необходимое
ensure_package("faiss-cpu", "faiss")
ensure_package("sentence-transformers", "sentence_transformers")
ensure_package("scikit-learn", "sklearn")

# Теперь основной импорт
import faiss
from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from IPython.display import display, Markdown

# Определяем устройство
try:
    import torch
    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
except ImportError:
    DEVICE = "cpu"

print(f"Устройство: {DEVICE}")
print(f"FAISS доступен: {faiss is not None}")

Устанавливаем пакет: faiss-cpu
Устанавливаем пакет: sentence-transformers
Устройство: cpu
FAISS доступен: True


In [2]:
# Ячейка 2
# Пример базы знаний (замени на свою тему!)
documents: List[Dict[str, str]] = [
    {
        "doc_id": "ml_01",
        "title": "Что такое машинное обучение",
        "text": "Машинное обучение — это подраздел искусственного интеллекта, который фокусируется на создании алгоритмов, способных обучаться на данных. Вместо того чтобы явно программировать правила, мы показываем модели множество примеров, и она сама находит закономерности. Например, чтобы научить компьютер отличать кошек от собак, мы даем ему тысячи фотографий кошек и собак."
    },
    {
        "doc_id": "ml_02",
        "title": "Обучение с учителем",
        "text": "Обучение с учителем — это самый распространенный тип машинного обучения. Здесь у нас есть данные, размеченные правильными ответами. Например, у нас есть таблица с информацией о домах (площадь, количество комнат) и их цена. Модель учится предсказывать цену на основе этих признаков. Ключевые задачи: регрессия (предсказание числа) и классификация (предсказание категории)."
    },
    {
        "doc_id": "ml_03",
        "title": "Обучение без учителя",
        "text": "В отличие от обучения с учителем, в обучении без учителя у нас нет правильных ответов. Цель — найти скрытые структуры в данных. Самый известный пример — кластеризация, когда мы группируем похожие объекты вместе. Например, мы можем сегментировать клиентов интернет-магазина на группы на основе истории их покупок."
    },
    {
        "doc_id": "ml_04",
        "title": "Переобучение и недообучение",
        "text": "Переобучение возникает, когда модель слишком хорошо запоминает обучающие данные, включая шум, и не может обобщить свои знания на новые, невиданные данные. Недообучение, наоборот, происходит, когда модель слишком проста, чтобы уловить закономерности в данных. Хорошая модель должна находить баланс между этими двумя крайностями."
    },
    {
        "doc_id": "ml_05",
        "title": "Что такое нейронная сеть",
        "text": "Искусственная нейронная сеть — это модель машинного обучения, вдохновленная строением мозга. Она состоит из слоев связанных между собой узлов, или 'нейронов'. Каждый нейрон принимает входные сигналы, обрабатывает их и передает результат дальше. Глубокое обучение использует нейронные сети с большим количеством слоев для решения очень сложных задач, таких как распознавание речи или вождение автомобиля."
    },
    # Добавь еще 5-15 документов в таком же формате
]

print(f"Количество документов в базе: {len(documents)}")
display(pd.DataFrame(documents)[["doc_id", "title"]].head())

Количество документов в базе: 5


,doc_id,title
0,ml_01,Что такое машинное обучение
1,ml_02,Обучение с учителем
2,ml_03,Обучение без учителя
3,ml_04,Переобучение и недообучение
4,ml_05,Что такое нейронная сеть


In [3]:
# Ячейка 3
def chunk_text(text: str, chunk_size: int = 300, overlap: int = 50) -> List[str]:
    """
    Простой чанкинг по символам.
    В реальности часто делают по токенам, но для старта сойдет и так.
    """
    chunks = []
    start = 0
    text_len = len(text)
    while start < text_len:
        end = start + chunk_size
        chunk = text[start:end]
        chunks.append(chunk)
        if end >= text_len:
            break
        start = end - overlap
    return chunks

# Параметры чанкинга (их можно будет менять в эксперименте)
CHUNK_SIZE = 300
OVERLAP = 50

# Создаем DataFrame с чанками
chunk_rows = []
for doc in documents:
    chunks = chunk_text(doc["text"], chunk_size=CHUNK_SIZE, overlap=OVERLAP)
    for i, chunk_text_val in enumerate(chunks):
        chunk_rows.append({
            "doc_id": doc["doc_id"],
            "title": doc["title"],
            "chunk_id": f"{doc['doc_id']}_chunk_{i:03d}",
            "chunk_text": chunk_text_val
        })

chunks_df = pd.DataFrame(chunk_rows)
print(f"Всего создано чанков: {len(chunks_df)}")
display(chunks_df.head(3))

Всего создано чанков: 10


,doc_id,title,chunk_id,chunk_text
0,ml_01,Что такое машинное обучение,ml_01_chunk_000,Машинное обучение — это подраздел искусственно...
1,ml_01,Что такое машинное обучение,ml_01_chunk_001,"омерности. Например, чтобы научить компьютер о..."
2,ml_02,Обучение с учителем,ml_02_chunk_000,Обучение с учителем — это самый распространенн...


In [4]:
# Ячейка 4
# Загружаем модель
model_name = 'sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2'
model = SentenceTransformer(model_name, device=DEVICE)

# Получаем эмбеддинги для всех чанков
chunk_texts = chunks_df["chunk_text"].tolist()
print("Вычисляем эмбеддинги для чанков...")
chunk_embeddings = model.encode(chunk_texts, normalize_embeddings=True, show_progress_bar=True)
print(f"Форма матрицы эмбеддингов: {chunk_embeddings.shape}")

# Строим FAISS индекс
dimension = chunk_embeddings.shape[1]
index = faiss.IndexFlatIP(dimension) # IP = Inner Product (скалярное произведение). Для норм. векторов == косинусное сходство
index.add(chunk_embeddings)
print(f"Индекс FAISS создан. Количество векторов в индексе: {index.ntotal}")

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

c:\Users\user\AppData\Local\Programs\Python\Python310\lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\user\.cache\huggingface\hub\models--sentence-transformers--paraphrase-multilingual-MiniLM-L12-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Вычисляем эмбеддинги для чанков...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Форма матрицы эмбеддингов: (10, 384)
Индекс FAISS создан. Количество векторов в индексе: 10


In [5]:
# Ячейка 5
def search(query: str, top_k: int = 3) -> pd.DataFrame:
    """Ищет top_k самых похожих чанков на запрос."""
    query_embedding = model.encode([query], normalize_embeddings=True)
    scores, indices = index.search(query_embedding, top_k)

    results = []
    for rank, (score, idx) in enumerate(zip(scores[0], indices[0]), start=1):
        chunk = chunks_df.iloc[idx].to_dict()
        chunk["rank"] = rank
        chunk["score"] = score
        results.append(chunk)

    return pd.DataFrame(results)[["rank", "score", "doc_id", "title", "chunk_id", "chunk_text"]]

# Проверим работу
sample_query = "Как нейронные сети связаны с мозгом?"
display(Markdown(f"### Запрос: {sample_query}"))
display(search(sample_query))

### Запрос: Как нейронные сети связаны с мозгом?

,rank,score,doc_id,title,chunk_id,chunk_text
0,1,0.801573,ml_05,Что такое нейронная сеть,ml_05_chunk_000,Искусственная нейронная сеть — это модель маши...
1,2,0.687789,ml_05,Что такое нейронная сеть,ml_05_chunk_001,кое обучение использует нейронные сети с больш...
2,3,0.313245,ml_03,Обучение без учителя,ml_03_chunk_000,"В отличие от обучения с учителем, в обучении б..."


In [7]:
# Ячейка 6
# ОЦЕНКА RETRIEVAL: Контрольные запросы
benchmark_queries = [
    {"query_id": "q01", "query": "Что такое машинное обучение?", "relevant_doc_id": "ml_01"},
    {"query_id": "q02", "query": "Как предсказать цену дома с помощью ML?", "relevant_doc_id": "ml_02"},
    {"query_id": "q03", "query": "Как сгруппировать клиентов по интересам?", "relevant_doc_id": "ml_03"},
    {"query_id": "q04", "query": "Почему модель отлично работает на обучающих данных, но плохо на новых?", "relevant_doc_id": "ml_04"},
    {"query_id": "q05", "query": "Что такое нейронная сеть?", "relevant_doc_id": "ml_05"},
    # ... добавь еще 5-7 запросов
]

def evaluate_retrieval(queries, k=3):
    eval_results = []
    for item in queries:
        query = item["query"]
        relevant_id = item["relevant_doc_id"]
        results_df = search(query, top_k=k)

        # hit@k
        hit = 1 if relevant_id in results_df["doc_id"].values else 0

        # recall@k (здесь он равен hit, т.к. у нас всегда один релевантный документ)
        recall = hit

        # rank of first relevant
        rank = results_df[results_df["doc_id"] == relevant_id]["rank"].values
        first_rank = rank[0] if len(rank) > 0 else None

        eval_results.append({
            "query_id": item["query_id"],
            "query": query,
            "expected_doc_id": relevant_id,
            "retrieved_doc_ids": ", ".join(results_df["doc_id"].values),
            f"hit@{k}": hit,
            f"recall@{k}": recall,
            "rank_of_first_relevant": first_rank
        })
    return pd.DataFrame(eval_results)

# Запускаем оценку для k=3
eval_df = evaluate_retrieval(benchmark_queries, k=3)
display(eval_df)

# Выводим итоговые метрики
print(f"Средний hit@3: {eval_df['hit@3'].mean():.2f}")
print(f"Средний recall@3: {eval_df['recall@3'].mean():.2f}")

# Сохраняем для артефакта
eval_df.to_csv("./artifacts/retrieval_eval.csv", index=False)
print("\nРезультаты сохранены в './artifacts/retrieval_eval.csv'")

,query_id,query,expected_doc_id,retrieved_doc_ids,hit@3,recall@3,rank_of_first_relevant
0,q01,Что такое машинное обучение?,ml_01,"ml_01, ml_02, ml_05",1,1,1
1,q02,Как предсказать цену дома с помощью ML?,ml_02,"ml_02, ml_02, ml_04",1,1,1
2,q03,Как сгруппировать клиентов по интересам?,ml_03,"ml_03, ml_03, ml_02",1,1,1
3,q04,Почему модель отлично работает на обучающих да...,ml_04,"ml_04, ml_02, ml_03",1,1,1
4,q05,Что такое нейронная сеть?,ml_05,"ml_05, ml_05, ml_01",1,1,1


Средний hit@3: 1.00
Средний recall@3: 1.00

Результаты сохранены в './artifacts/retrieval_eval.csv'


In [8]:
# Ячейка 7
def run_experiment(chunk_size):
    print(f"\n--- Запуск эксперимента с chunk_size={chunk_size} ---")
    # Пересоздаем чанки
    temp_chunks = []
    for doc in documents:
        chunks = chunk_text(doc["text"], chunk_size=chunk_size, overlap=50)
        for i, chunk_text_val in enumerate(chunks):
            temp_chunks.append({
                "doc_id": doc["doc_id"],
                "title": doc["title"],
                "chunk_id": f"{doc['doc_id']}_chunk_{i:03d}",
                "chunk_text": chunk_text_val
            })
    temp_chunks_df = pd.DataFrame(temp_chunks)
    temp_texts = temp_chunks_df["chunk_text"].tolist()
    temp_embeddings = model.encode(temp_texts, normalize_embeddings=True, show_progress_bar=False)

    # Пересоздаем индекс
    temp_index = faiss.IndexFlatIP(temp_embeddings.shape[1])
    temp_index.add(temp_embeddings)

    # Временно подменяем глобальные переменные для функции search
    global chunks_df, index
    original_chunks_df, original_index = chunks_df, index
    chunks_df, index = temp_chunks_df, temp_index

    # Запускаем оценку
    eval_df = evaluate_retrieval(benchmark_queries, k=3)

    # Возвращаем как было
    chunks_df, index = original_chunks_df, original_index

    return {
        "chunk_size": chunk_size,
        "num_chunks": len(temp_chunks_df),
        "mean_hit@3": eval_df['hit@3'].mean(),
        "mean_recall@3": eval_df['recall@3'].mean()
    }

# Запускаем для двух вариантов
exp_results = [run_experiment(200), run_experiment(400)]
exp_df = pd.DataFrame(exp_results)
display(exp_df)

# ВЫВОД: Смотрим, какой chunk_size дал лучший результат, и выбираем его как основной.
# Допустим, chunk_size=300 (наш базовый) показал себя лучше или так же. Оставляем 300.


--- Запуск эксперимента с chunk_size=200 ---

--- Запуск эксперимента с chunk_size=400 ---


,chunk_size,num_chunks,mean_hit@3,mean_recall@3
0,200,13,1.0,1.0
1,400,6,1.0,1.0


In [9]:
# Ячейка 8
# 1. Сохраняем результаты ДО обновления для сравнения
queries_for_comparison = [
    "Что такое градиентный бустинг?",
    "Зачем нужна функция активации в нейронной сети?",
    "Как работает метод опорных векторов (SVM)?"
]

print("=== РЕЗУЛЬТАТЫ ДО ОБНОВЛЕНИЯ ===")
before_results = {}
for q in queries_for_comparison:
    res_df = search(q, top_k=3)
    before_results[q] = ", ".join(res_df["doc_id"].values)
    print(f"\nЗапрос: {q}\nНайденные ID: {before_results[q]}")
    display(res_df[["rank", "score", "doc_id", "chunk_text"]])


# 2. Добавляем новые документы
new_documents = [
    {
        "doc_id": "ml_06",
        "title": "Градиентный бустинг",
        "text": "Градиентный бустинг — это мощная техника машинного обучения, которая строит модель в виде ансамбля слабых предсказательных моделей, чаще всего деревьев решений. Он строит модель поэтапно и позволяет оптимизировать произвольную дифференцируемую функцию потерь."
    },
    {
        "doc_id": "ml_07",
        "title": "Функции активации",
        "text": "Функция активации в нейронной сети определяет выходной сигнал нейрона на основе его входа. Она вносит нелинейность в модель, позволяя ей обучаться сложным зависимостям. Без функций активации нейронная сеть была бы просто линейной регрессией. Популярные функции: ReLU, сигмоида, tanh."
    },
    {
        "doc_id": "ml_08",
        "title": "Метод опорных векторов (SVM)",
        "text": "Метод опорных векторов (SVM) — это алгоритм обучения с учителем, используемый для задач классификации и регрессии. Основная идея SVM — найти гиперплоскость, которая наилучшим образом разделяет данные на классы с максимальным зазором."
    }
]
documents.extend(new_documents)
print("\n\nДокументы добавлены. Переиндексация...")

# 3. Переиндексация (повторяем Шаг 3 и 4)
chunk_rows = []
for doc in documents:
    chunks = chunk_text(doc["text"], chunk_size=CHUNK_SIZE, overlap=OVERLAP)
    for i, chunk_text_val in enumerate(chunks):
        chunk_rows.append({
            "doc_id": doc["doc_id"],
            "title": doc["title"],
            "chunk_id": f"{doc['doc_id']}_chunk_{i:03d}",
            "chunk_text": chunk_text_val
        })
chunks_df = pd.DataFrame(chunk_rows)
chunk_texts = chunks_df["chunk_text"].tolist()
chunk_embeddings = model.encode(chunk_texts, normalize_embeddings=True, show_progress_bar=True)
index = faiss.IndexFlatIP(chunk_embeddings.shape[1])
index.add(chunk_embeddings)
print("Переиндексация завершена.")

# 4. Сравниваем ПОСЛЕ
print("\n=== РЕЗУЛЬТАТЫ ПОСЛЕ ОБНОВЛЕНИЯ ===")
comparison_data = []
for q in queries_for_comparison:
    res_df = search(q, top_k=3)
    after_doc_ids = ", ".join(res_df["doc_id"].values)
    print(f"\nЗапрос: {q}\nНайденные ID: {after_doc_ids}")
    display(res_df[["rank", "score", "doc_id", "chunk_text"]])

    comparison_data.append({
        "query": q,
        "before_retrieved_doc_ids": before_results[q],
        "after_retrieved_doc_ids": after_doc_ids,
        "changed": before_results[q] != after_doc_ids
    })

# 5. Сохраняем сравнение в артефакт
comparison_df = pd.DataFrame(comparison_data)
comparison_df.to_csv("./artifacts/retrieval_before_after_update.csv", index=False)
print("\nРезультаты сравнения сохранены в './artifacts/retrieval_before_after_update.csv'")
display(comparison_df)

=== РЕЗУЛЬТАТЫ ДО ОБНОВЛЕНИЯ ===

Запрос: Что такое градиентный бустинг?
Найденные ID: ml_04, ml_05, ml_05


,rank,score,doc_id,chunk_text
0,1,0.301658,ml_04,данных. Хорошая модель должна находить баланс...
1,2,0.291992,ml_05,Искусственная нейронная сеть — это модель маши...
2,3,0.253500,ml_05,кое обучение использует нейронные сети с больш...



Запрос: Зачем нужна функция активации в нейронной сети?
Найденные ID: ml_05, ml_05, ml_04


,rank,score,doc_id,chunk_text
0,1,0.762808,ml_05,Искусственная нейронная сеть — это модель маши...
1,2,0.698970,ml_05,кое обучение использует нейронные сети с больш...
2,3,0.357562,ml_04,"Переобучение возникает, когда модель слишком х..."



Запрос: Как работает метод опорных векторов (SVM)?
Найденные ID: ml_01, ml_02, ml_05


...

,rank,score,doc_id,chunk_text
0,1,0.869200,ml_07,Функция активации в нейронной сети определяет ...
1,2,0.762809,ml_05,Искусственная нейронная сеть — это модель маши...
2,3,0.698970,ml_05,кое обучение использует нейронные сети с больш...



Запрос: Как работает метод опорных векторов (SVM)?
Найденные ID: ml_08, ml_06, ml_01


,rank,score,doc_id,chunk_text
0,1,0.842087,ml_08,Метод опорных векторов (SVM) — это алгоритм об...
1,2,0.376209,ml_06,Градиентный бустинг — это мощная техника машин...
2,3,0.298174,ml_01,Машинное обучение — это подраздел искусственно...



Результаты сравнения сохранены в './artifacts/retrieval_before_after_update.csv'


,query,before_retrieved_doc_ids,after_retrieved_doc_ids,changed
0,Что такое градиентный бустинг?,"ml_04, ml_05, ml_05","ml_06, ml_04, ml_08",True
1,Зачем нужна функция активации в нейронной сети?,"ml_05, ml_05, ml_04","ml_07, ml_05, ml_05",True
2,Как работает метод опорных векторов (SVM)?,"ml_01, ml_02, ml_05","ml_08, ml_06, ml_01",True


In [10]:
# Ячейка 9
import re
def split_into_sentences(text):
    return re.split(r'(?<=[.!?])\s+', text)

def mini_rag_answer(query: str, top_k: int = 3) -> Dict:
    retrieved_df = search(query, top_k=top_k)

    # Собираем контекст из найденных чанков
    context = "\n\n".join([f"[{row['doc_id']}] {row['chunk_text']}" for _, row in retrieved_df.iterrows()])

    # "Генерация" ответа: берем первое предложение из самого первого (самого релевантного) чанка
    # Это очень наивный, но наглядный extractive-подход.
    top_chunk_text = retrieved_df.iloc[0]['chunk_text']
    answer_sentence = split_into_sentences(top_chunk_text)[0]

    return {
        "question": query,
        "answer": answer_sentence,
        "retrieved_sources": ", ".join(retrieved_df["doc_id"].unique()), # уникальные источники
        "context": context # можно не выводить, но для отладки полезно
    }

# Пример работы RAG
rag_examples = []
test_questions = [
    "Что такое переобучение?",
    "Расскажи про нейронные сети.",
    "Какие есть типы машинного обучения?"
]

for q in test_questions:
    rag_result = mini_rag_answer(q)
    print(f"Вопрос: {q}")
    print(f"Ответ: {rag_result['answer']}")
    print(f"Источники: {rag_result['retrieved_sources']}")
    print("-" * 50)

    rag_examples.append({
        "question": q,
        "answer": rag_result['answer'],
        "retrieved_sources": rag_result['retrieved_sources']
    })

# Сохраняем в артефакт
rag_examples_df = pd.DataFrame(rag_examples)
rag_examples_df.to_csv("./artifacts/rag_examples.csv", index=False)
print("Примеры RAG сохранены в './artifacts/rag_examples.csv'")

Вопрос: Что такое переобучение?
Ответ: Переобучение возникает, когда модель слишком хорошо запоминает обучающие данные, включая шум, и не может обобщить свои знания на новые, невиданные данные.
Источники: ml_04, ml_06, ml_05
--------------------------------------------------
Вопрос: Расскажи про нейронные сети.
Ответ: Искусственная нейронная сеть — это модель машинного обучения, вдохновленная строением мозга.
Источники: ml_05, ml_07
--------------------------------------------------
Вопрос: Какие есть типы машинного обучения?
Ответ: Машинное обучение — это подраздел искусственного интеллекта, который фокусируется на создании алгоритмов, способных обучаться на данных.
Источники: ml_01, ml_02, ml_06
--------------------------------------------------
Примеры RAG сохранены в './artifacts/rag_examples.csv'
